# 🏅 Olympic Data Lake Pipeline

### Camadas:
- **RAW**: Dados brutos originais (CSV + metadados JSON)
- **BRONZE**: Dados padronizados (Parquet + limpeza básica)
- **PRATA**: Dados integrados (JOINs + curadoria)
- **GOLD**: Dados analíticos (agregações + visualizações)

---
## 0. Setup e Importações

In [1]:
import pandas as pd
import numpy as np
import json
import os
import re
import shutil
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.use('Agg')  # backend não-interativo para salvar PNGs
sns.set_theme(style="whitegrid")

# Diretório base do projeto (onde este notebook está)
BASE_DIR = Path.cwd()
DATALAKE_DIR = BASE_DIR / "olympics-datalake"

# Diretório dos dados brutos originais
RAW_SOURCE_DIR = BASE_DIR / "raw"

# Camadas do Data Lake
RAW_DIR = DATALAKE_DIR / "raw"
BRONZE_DIR = DATALAKE_DIR / "bronze"
PRATA_DIR = DATALAKE_DIR / "prata"
GOLD_DIR = DATALAKE_DIR / "gold"

# Timestamp de processamento
PROCESSING_DATE = datetime.now().isoformat()

print(f"Base dir: {BASE_DIR}")
print(f"Data Lake dir: {DATALAKE_DIR}")
print(f"Processing date: {PROCESSING_DATE}")

Base dir: e:\code\study\uea\Ciencia_de_Dados\trabalho_3
Data Lake dir: e:\code\study\uea\Ciencia_de_Dados\trabalho_3\olympics-datalake
Processing date: 2026-03-30T02:50:31.647504


### Funções Utilitárias

In [2]:
def ensure_dir(path: Path) -> Path:
    """Cria diretório se não existir (idempotente)."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def normalize_column_name(col: str) -> str:
    """Converte nome de coluna para snake_case minúsculo."""
    col = col.strip().lower()
    col = re.sub(r'[^a-z0-9]+', '_', col)
    col = col.strip('_')
    return col


def save_metadata(metadata: dict, filepath: Path):
    """Salva metadados como JSON formatado."""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f"  ✓ Metadata salvo: {filepath.relative_to(DATALAKE_DIR)}")


def get_schema_info(df: pd.DataFrame) -> list:
    """Retorna schema (colunas + tipos) de um DataFrame."""
    return [
        {"column": col, "dtype": str(dtype), "non_null_count": int(df[col].notna().sum())}
        for col, dtype in df.dtypes.items()
    ]


def load_csv_safe(filepath: Path) -> pd.DataFrame:
    """Carrega CSV tratando problemas de encoding."""
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            df = pd.read_csv(filepath, encoding=enc, low_memory=False)
            print(f"  ✓ Carregado com encoding '{enc}': {df.shape[0]} linhas, {df.shape[1]} colunas")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"Não foi possível ler {filepath} com nenhum encoding suportado")


def validate_schema(df: pd.DataFrame, name: str):
    """Valida consistência básica de schema."""
    issues = []
    # Verifica colunas duplicadas
    dup_cols = df.columns[df.columns.duplicated()].tolist()
    if dup_cols:
        issues.append(f"Colunas duplicadas: {dup_cols}")
    # Verifica colunas totalmente nulas
    all_null = [col for col in df.columns if df[col].isna().all()]
    if all_null:
        issues.append(f"Colunas 100% nulas: {all_null}")
    # Verifica linhas totalmente duplicadas
    dup_rows = df.duplicated().sum()
    if dup_rows > 0:
        issues.append(f"{dup_rows} linhas duplicadas encontradas")

    if issues:
        print(f"  ⚠ Validação '{name}': {'; '.join(issues)}")
    else:
        print(f"  ✓ Validação '{name}': schema consistente")
    return issues


print("Funções utilitárias carregadas.")

Funções utilitárias carregadas.


### Criar Estrutura de Diretórios

In [3]:
# Criar todas as pastas do Data Lake
dirs = [
    RAW_DIR,
    BRONZE_DIR,
    PRATA_DIR,
    GOLD_DIR / "analise_medalhas",
    GOLD_DIR / "analise_modalidades",
    GOLD_DIR / "analise_genero",
]
for d in dirs:
    ensure_dir(d)
    print(f"✓ {d.relative_to(DATALAKE_DIR)}")

print("\nEstrutura de diretórios criada com sucesso.")

✓ raw
✓ bronze
✓ prata
✓ gold\analise_medalhas
✓ gold\analise_modalidades
✓ gold\analise_genero

Estrutura de diretórios criada com sucesso.


---
## 1. Camada RAW — Ingestão de Dados Brutos

Nesta etapa:
- Carregamos os CSVs originais
- Validamos a consistência de schema
- Copiamos para a camada RAW do Data Lake
- Criamos metadados JSON para cada dataset

In [4]:
# Mapeamento dos datasets originais
RAW_DATASETS = {
    "olympics_historico": {
        "files": {
            "athlete_event_result": RAW_SOURCE_DIR / "1986-2022" / "world_olympedia_olympics_athlete_event_result.csv",
            "game_medal_tally": RAW_SOURCE_DIR / "1986-2022" / "world_olympedia_olympics_game_medal_tally(4).csv",
            "result": RAW_SOURCE_DIR / "1986-2022" / "world_olympedia_olympics_result(2).csv",
        },
        "source": "Olympedia / World Olympic Data (1896-2022)",
        "description": "Dados históricos de Olimpíadas incluindo resultados de atletas, quadro de medalhas por país e detalhes de eventos.",
    },
    "olympics_paris2024": {
        "files": {
            "medallists": RAW_SOURCE_DIR / "2024" / "medallists.csv",
            "medals": RAW_SOURCE_DIR / "2024" / "medals(1).csv",
        },
        "source": "Kaggle / Paris 2024 Olympics Dataset",
        "description": "Dados das Olimpíadas de Paris 2024 incluindo medalhistas individuais e quadro de medalhas.",
    },
}

print("Datasets mapeados:")
for group, info in RAW_DATASETS.items():
    print(f"  {group}: {list(info['files'].keys())}")

Datasets mapeados:
  olympics_historico: ['athlete_event_result', 'game_medal_tally', 'result']
  olympics_paris2024: ['medallists', 'medals']


In [5]:
# Carregar e validar cada dataset
raw_dataframes = {}

for group_name, group_info in RAW_DATASETS.items():
    print(f"\n{'='*60}")
    print(f"Processando grupo: {group_name}")
    print(f"{'='*60}")

    for ds_name, filepath in group_info["files"].items():
        key = f"{group_name}__{ds_name}"
        print(f"\n→ Dataset: {ds_name}")
        print(f"  Arquivo: {filepath.name}")

        # Carregar CSV
        df = load_csv_safe(filepath)

        # Validar schema
        validate_schema(df, ds_name)

        # Armazenar
        raw_dataframes[key] = df

print(f"\n✓ Total de datasets carregados: {len(raw_dataframes)}")


Processando grupo: olympics_historico

→ Dataset: athlete_event_result
  Arquivo: world_olympedia_olympics_athlete_event_result.csv
  ✓ Carregado com encoding 'utf-8': 316834 linhas, 11 colunas
  ⚠ Validação 'athlete_event_result': 1208 linhas duplicadas encontradas

→ Dataset: game_medal_tally
  Arquivo: world_olympedia_olympics_game_medal_tally(4).csv
  ✓ Carregado com encoding 'utf-8': 1807 linhas, 9 colunas
  ✓ Validação 'game_medal_tally': schema consistente

→ Dataset: result
  Arquivo: world_olympedia_olympics_result(2).csv
  ✓ Carregado com encoding 'utf-8': 7394 linhas, 11 colunas
  ✓ Validação 'result': schema consistente

Processando grupo: olympics_paris2024

→ Dataset: medallists
  Arquivo: medallists.csv
  ✓ Carregado com encoding 'utf-8': 2315 linhas, 21 colunas
  ✓ Validação 'medallists': schema consistente

→ Dataset: medals
  Arquivo: medals(1).csv
  ✓ Carregado com encoding 'utf-8': 1044 linhas, 13 colunas
  ✓ Validação 'medals': schema consistente

✓ Total de datas

In [6]:
# Consolidar CSVs por grupo e salvar na camada RAW
# Histórico: usamos o athlete_event_result como dataset principal
# Paris 2024: usamos o medallists como dataset principal

# Salvar CSVs individuais na camada RAW
raw_csv_files = {
    "olympics_historico": raw_dataframes["olympics_historico__athlete_event_result"],
    "olympics_paris2024": raw_dataframes["olympics_paris2024__medallists"],
}

for name, df in raw_csv_files.items():
    csv_path = RAW_DIR / f"{name}.csv"
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✓ Salvo: {csv_path.relative_to(DATALAKE_DIR)}")

# Também salvar como JSON (conforme requisito)
for name, df in raw_csv_files.items():
    json_path = RAW_DIR / f"{name}.json"
    df.to_json(json_path, orient='records', force_ascii=False, indent=2)
    print(f"✓ Salvo: {json_path.relative_to(DATALAKE_DIR)}")

# Salvar também os datasets auxiliares
aux_datasets = {
    "medal_tally_historico": raw_dataframes["olympics_historico__game_medal_tally"],
    "result_historico": raw_dataframes["olympics_historico__result"],
    "medals_paris2024": raw_dataframes["olympics_paris2024__medals"],
}
for name, df in aux_datasets.items():
    csv_path = RAW_DIR / f"{name}.csv"
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✓ Salvo (auxiliar): {csv_path.relative_to(DATALAKE_DIR)}")

✓ Salvo: raw\olympics_historico.csv
✓ Salvo: raw\olympics_paris2024.csv
✓ Salvo: raw\olympics_historico.json
✓ Salvo: raw\olympics_paris2024.json
✓ Salvo (auxiliar): raw\medal_tally_historico.csv
✓ Salvo (auxiliar): raw\result_historico.csv
✓ Salvo (auxiliar): raw\medals_paris2024.csv


In [7]:
# Gerar metadados JSON para cada dataset RAW
raw_metadata_definitions = {
    "olympics_historico": {
        "dataset_name": "olympics_historico",
        "source": "Olympedia / World Olympic Data",
        "description": "Resultados individuais de atletas em eventos olímpicos de 1896 a 2022, incluindo edição, país, esporte, evento, posição e medalha.",
        "main_fields": ["edition", "country_noc", "sport", "event", "athlete", "medal", "position"],
        "collection_date": "2023-01-01",
        "observations": "Campo 'medal' possui ~86% de valores nulos (atletas sem medalha). Dados cobrem Olimpíadas de Verão e Inverno."
    },
    "olympics_paris2024": {
        "dataset_name": "olympics_paris2024",
        "source": "Kaggle / Paris 2024 Olympics Dataset",
        "description": "Medalhistas individuais das Olimpíadas de Paris 2024, incluindo dados de disciplina, evento, país e informações do atleta.",
        "main_fields": ["medal_type", "name", "gender", "country_code", "discipline", "event"],
        "collection_date": "2024-08-15",
        "observations": "Campos 'team', 'team_gender' e 'code_team' são nulos para eventos individuais (~33% dos registros)."
    },
    "medal_tally_historico": {
        "dataset_name": "medal_tally_historico",
        "source": "Olympedia / World Olympic Data",
        "description": "Quadro de medalhas por país e edição olímpica de 1896 a 2022.",
        "main_fields": ["year", "edition", "country", "country_noc", "gold", "silver", "bronze", "total"],
        "collection_date": "2023-01-01",
        "observations": "1807 registros sem valores nulos. Dados completos e consistentes."
    },
    "result_historico": {
        "dataset_name": "result_historico",
        "source": "Olympedia / World Olympic Data",
        "description": "Detalhes de resultados de eventos olímpicos incluindo localização, formato e descrição.",
        "main_fields": ["result_id", "event_title", "edition", "sport", "result_date", "result_location"],
        "collection_date": "2023-01-01",
        "observations": "Campo 'result_location' possui 1 valor nulo. Campo 'result_description' pode conter textos longos."
    },
    "medals_paris2024": {
        "dataset_name": "medals_paris2024",
        "source": "Kaggle / Paris 2024 Olympics Dataset",
        "description": "Quadro de medalhas das Olimpíadas de Paris 2024 com tipo, data, atleta, disciplina e país.",
        "main_fields": ["medal_type", "medal_date", "name", "gender", "discipline", "event", "country_code"],
        "collection_date": "2024-08-15",
        "observations": "1044 registros. Campo 'url_event' possui 9 nulos."
    },
}

for ds_name, meta_base in raw_metadata_definitions.items():
    # Buscar o DataFrame correspondente
    if ds_name in raw_csv_files:
        df = raw_csv_files[ds_name]
    else:
        df = aux_datasets[ds_name]

    metadata = {
        **meta_base,
        "layer": "raw",
        "schema": get_schema_info(df),
        "row_count": len(df),
        "column_count": len(df.columns),
        "processing_date": PROCESSING_DATE,
    }
    save_metadata(metadata, RAW_DIR / f"{ds_name}_metadata.json")

print("\n✓ Camada RAW concluída.")

  ✓ Metadata salvo: raw\olympics_historico_metadata.json
  ✓ Metadata salvo: raw\olympics_paris2024_metadata.json
  ✓ Metadata salvo: raw\medal_tally_historico_metadata.json
  ✓ Metadata salvo: raw\result_historico_metadata.json
  ✓ Metadata salvo: raw\medals_paris2024_metadata.json

✓ Camada RAW concluída.


---
## 2. Camada BRONZE — Padronização de Dados

Nesta etapa:
- Convertemos CSVs → Parquet
- Normalizamos nomes de colunas (snake_case)
- Aplicamos limpeza básica (trimming de strings, tratamento de nulos)
- Geramos metadados com transformações aplicadas

In [8]:
def process_bronze(df: pd.DataFrame, dataset_name: str) -> tuple:
    """Aplica transformações da camada Bronze e retorna (df_limpo, lista_transformações)."""
    transformations = []

    # 1. Normalizar nomes de colunas
    original_cols = list(df.columns)
    df.columns = [normalize_column_name(c) for c in df.columns]
    renamed = {o: n for o, n in zip(original_cols, df.columns) if o != n}
    if renamed:
        transformations.append(f"Colunas renomeadas: {renamed}")

    # 2. Trimming de strings
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
        # Converter 'nan' string de volta para NaN
        df[col] = df[col].replace('nan', np.nan)
    if len(str_cols) > 0:
        transformations.append(f"Trimming aplicado em {len(str_cols)} colunas string")

    # 3. Remover linhas 100% duplicadas
    n_dups = df.duplicated().sum()
    if n_dups > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        transformations.append(f"{n_dups} linhas duplicadas removidas")

    # 4. Padronizar colunas booleanas
    for col in df.columns:
        if df[col].dtype == 'object':
            unique_vals = set(df[col].dropna().str.lower().unique())
            if unique_vals.issubset({'true', 'false'}):
                df[col] = df[col].str.lower().map({'true': True, 'false': False})
                transformations.append(f"Coluna '{col}' convertida para boolean")

    return df, transformations


print("Função process_bronze definida.")

Função process_bronze definida.


In [9]:
# Todos os datasets para processar na camada Bronze
all_raw = {**raw_csv_files, **aux_datasets}
bronze_dataframes = {}

for name, df_raw in all_raw.items():
    print(f"\n{'='*50}")
    print(f"Bronze: {name}")
    print(f"{'='*50}")

    df = df_raw.copy()
    df_bronze, transforms = process_bronze(df, name)

    # Salvar como Parquet
    parquet_path = BRONZE_DIR / f"{name}.parquet"
    df_bronze.to_parquet(parquet_path, index=False, engine='pyarrow')
    print(f"  ✓ Parquet salvo: {parquet_path.relative_to(DATALAKE_DIR)}")
    print(f"  Shape: {df_bronze.shape}")

    # Gerar metadados
    metadata = {
        "dataset_name": name,
        "layer": "bronze",
        "source_dataset": f"raw/{name}.csv",
        "schema": get_schema_info(df_bronze),
        "transformations_applied": transforms,
        "row_count": len(df_bronze),
        "column_count": len(df_bronze.columns),
        "processing_date": PROCESSING_DATE,
    }
    save_metadata(metadata, BRONZE_DIR / f"{name}_metadata.json")

    bronze_dataframes[name] = df_bronze

print(f"\n✓ Camada BRONZE concluída. {len(bronze_dataframes)} datasets processados.")


Bronze: olympics_historico
  ✓ Parquet salvo: bronze\olympics_historico.parquet
  Shape: (315626, 11)
  ✓ Metadata salvo: bronze\olympics_historico_metadata.json

Bronze: olympics_paris2024
  ✓ Parquet salvo: bronze\olympics_paris2024.parquet
  Shape: (2315, 21)
  ✓ Metadata salvo: bronze\olympics_paris2024_metadata.json

Bronze: medal_tally_historico
  ✓ Parquet salvo: bronze\medal_tally_historico.parquet
  Shape: (1807, 9)
  ✓ Metadata salvo: bronze\medal_tally_historico_metadata.json

Bronze: result_historico
  ✓ Parquet salvo: bronze\result_historico.parquet
  Shape: (7394, 11)
  ✓ Metadata salvo: bronze\result_historico_metadata.json

Bronze: medals_paris2024
  ✓ Parquet salvo: bronze\medals_paris2024.parquet
  Shape: (1044, 13)
  ✓ Metadata salvo: bronze\medals_paris2024_metadata.json

✓ Camada BRONZE concluída. 5 datasets processados.


In [10]:
# Inspecionar schemas da camada Bronze
for name, df in bronze_dataframes.items():
    print(f"\n{name}:")
    print(f"  Colunas: {list(df.columns)}")
    print(f"  Shape: {df.shape}")


olympics_historico:
  Colunas: ['edition', 'edition_id', 'country_noc', 'sport', 'event', 'result_id', 'athlete', 'athlete_id', 'position', 'medal', 'is_team_sport']
  Shape: (315626, 11)

olympics_paris2024:
  Colunas: ['medal_date', 'medal_type', 'medal_code', 'name', 'gender', 'country_code', 'country', 'country_long', 'nationality_code', 'nationality', 'nationality_long', 'team', 'team_gender', 'discipline', 'event', 'event_type', 'url_event', 'birth_date', 'code_athlete', 'code_team', 'is_medallist']
  Shape: (2315, 21)

medal_tally_historico:
  Colunas: ['year', 'edition', 'edition_id', 'country', 'country_noc', 'gold', 'silver', 'bronze', 'total']
  Shape: (1807, 9)

result_historico:
  Colunas: ['result_id', 'event_title', 'edition', 'edition_id', 'sport', 'result_date', 'result_location', 'result_participants', 'result_format', 'result_detail', 'result_description']
  Shape: (7394, 11)

medals_paris2024:
  Colunas: ['medal_type', 'medal_code', 'medal_date', 'name', 'gender', 

---
## 3. Camada PRATA (SILVER) — Integração de Dados

Nesta etapa criamos datasets curados:
1. **medalhas_1986_2024**: Quadro de medalhas unificado (histórico + Paris 2024)
2. **modalidades_1986_2024**: Esportes/eventos ao longo dos anos
3. **atletas_por_sexo**: Participação e medalhas por gênero

### 3.1 Dataset: medalhas_1986_2024

Unificação do quadro de medalhas por país: dados históricos (1896–2022) + Paris 2024.

In [11]:
# Histórico: usar medal_tally_historico (já tem gold/silver/bronze por país/edição)
df_tally_hist = bronze_dataframes["medal_tally_historico"].copy()
print(f"Medal tally histórico: {df_tally_hist.shape}")
print(f"Colunas: {list(df_tally_hist.columns)}")
df_tally_hist.head(3)

Medal tally histórico: (1807, 9)
Colunas: ['year', 'edition', 'edition_id', 'country', 'country_noc', 'gold', 'silver', 'bronze', 'total']


,year,edition,edition_id,country,country_noc,gold,silver,bronze,total
0,1896,1896 Summer Olympics,1,Greece,GRE,10,18,19,47
1,1900,1900 Summer Olympics,2,France,FRA,31,41,40,112
2,1900,1900 Summer Olympics,2,United States,USA,20,13,15,48


In [12]:
# Paris 2024: agregar medalhas por país a partir de medals_paris2024
df_medals_2024 = bronze_dataframes["medals_paris2024"].copy()

# Criar colunas gold/silver/bronze
df_medals_2024['gold'] = (df_medals_2024['medal_type'].str.lower().str.contains('gold')).astype(int)
df_medals_2024['silver'] = (df_medals_2024['medal_type'].str.lower().str.contains('silver')).astype(int)
df_medals_2024['bronze_medal'] = (df_medals_2024['medal_type'].str.lower().str.contains('bronze')).astype(int)

# Agregar por país
tally_2024 = df_medals_2024.groupby(['country_code', 'country', 'country_long']).agg(
    gold=('gold', 'sum'),
    silver=('silver', 'sum'),
    bronze=('bronze_medal', 'sum'),
).reset_index()
tally_2024['total'] = tally_2024['gold'] + tally_2024['silver'] + tally_2024['bronze']
tally_2024['year'] = 2024
tally_2024['edition'] = '2024 Summer Olympics'
tally_2024['edition_id'] = 9999  # placeholder

# Renomear para compatibilidade com histórico
tally_2024 = tally_2024.rename(columns={'country_code': 'country_noc', 'country_long': 'country'})
tally_2024 = tally_2024.drop(columns=['country'], errors='ignore')
tally_2024 = tally_2024.rename(columns={'country': 'country'})

print(f"Medal tally Paris 2024: {tally_2024.shape}")
tally_2024.sort_values('total', ascending=False).head(5)

Medal tally Paris 2024: (92, 8)


,country_noc,gold,silver,bronze,total,year,edition,edition_id
89,USA,40,44,42,126,2024,2024 Summer Olympics,9999
15,CHN,40,27,24,91,2024,2024 Summer Olympics,9999
33,GBR,14,22,29,65,2024,2024 Summer Olympics,9999
32,FRA,16,26,22,64,2024,2024 Summer Olympics,9999
5,AUS,18,19,16,53,2024,2024 Summer Olympics,9999


In [13]:
# Unificar: selecionar colunas comuns
common_cols = ['year', 'edition', 'edition_id', 'country_noc', 'gold', 'silver', 'bronze', 'total']

# Adicionar coluna 'country' ao histórico se existir
hist_cols = [c for c in common_cols if c in df_tally_hist.columns]
tally_2024_cols = [c for c in common_cols if c in tally_2024.columns]

# Se a coluna 'country' existe no histórico, incluí-la
if 'country' in df_tally_hist.columns:
    hist_cols = ['country'] + hist_cols if 'country' not in hist_cols else hist_cols

df_medalhas = pd.concat([
    df_tally_hist[hist_cols],
    tally_2024[tally_2024_cols],
], ignore_index=True)

# Deduplicação
before = len(df_medalhas)
df_medalhas = df_medalhas.drop_duplicates().reset_index(drop=True)
after = len(df_medalhas)
print(f"medalhas_1986_2024: {after} linhas (removidas {before - after} duplicatas)")

# Salvar
prata_path = PRATA_DIR / "medalhas_1986_2024.parquet"
df_medalhas.to_parquet(prata_path, index=False, engine='pyarrow')
print(f"✓ Salvo: {prata_path.relative_to(DATALAKE_DIR)}")

df_medalhas.head()

medalhas_1986_2024: 1899 linhas (removidas 0 duplicatas)
✓ Salvo: prata\medalhas_1986_2024.parquet


,country,year,edition,edition_id,country_noc,gold,silver,bronze,total
0,Greece,1896,1896 Summer Olympics,1,GRE,10,18,19,47
1,France,1900,1900 Summer Olympics,2,FRA,31,41,40,112
2,United States,1900,1900 Summer Olympics,2,USA,20,13,15,48
3,United States,1904,1904 Summer Olympics,3,USA,80,85,83,248
4,Great Britain,1908,1908 Summer Olympics,5,GBR,56,51,39,146


### 3.2 Dataset: modalidades_1986_2024

Esportes e eventos olímpicos ao longo do tempo — dados históricos + Paris 2024.

In [14]:
# Histórico: extrair sport/event/edition do athlete_event_result
df_hist = bronze_dataframes["olympics_historico"].copy()

modalidades_hist = df_hist[['edition', 'edition_id', 'sport', 'event']].drop_duplicates()
# Extrair ano
modalidades_hist['year'] = modalidades_hist['edition'].str.extract(r'(\d{4})').astype(int)
modalidades_hist['source'] = 'historico'
print(f"Modalidades históricas: {len(modalidades_hist)} combinações únicas sport/event/edition")

# Paris 2024: extrair discipline/event do medallists
df_2024 = bronze_dataframes["olympics_paris2024"].copy()

modalidades_2024 = df_2024[['discipline', 'event']].drop_duplicates()
modalidades_2024 = modalidades_2024.rename(columns={'discipline': 'sport'})
modalidades_2024['year'] = 2024
modalidades_2024['edition'] = '2024 Summer Olympics'
modalidades_2024['edition_id'] = 9999
modalidades_2024['source'] = 'paris2024'
print(f"Modalidades Paris 2024: {len(modalidades_2024)} combinações únicas")

# Unificar
df_modalidades = pd.concat([
    modalidades_hist[['year', 'edition', 'edition_id', 'sport', 'event', 'source']],
    modalidades_2024[['year', 'edition', 'edition_id', 'sport', 'event', 'source']],
], ignore_index=True)

df_modalidades = df_modalidades.drop_duplicates().reset_index(drop=True)
print(f"\nmodalidades_1986_2024: {len(df_modalidades)} registros")

# Salvar
prata_path = PRATA_DIR / "modalidades_1986_2024.parquet"
df_modalidades.to_parquet(prata_path, index=False, engine='pyarrow')
print(f"✓ Salvo: {prata_path.relative_to(DATALAKE_DIR)}")

df_modalidades.head()

Modalidades históricas: 7102 combinações únicas sport/event/edition
Modalidades Paris 2024: 329 combinações únicas

modalidades_1986_2024: 7431 registros
✓ Salvo: prata\modalidades_1986_2024.parquet


,year,edition,edition_id,sport,event,source
0,1928,1928 Winter Olympics,30,Skeleton,"Skeleton, Men",historico
1,1948,1948 Winter Olympics,33,Skeleton,"Skeleton, Men",historico
2,2002,2002 Winter Olympics,47,Skeleton,"Skeleton, Men",historico
3,2002,2002 Winter Olympics,47,Skeleton,"Skeleton, Women",historico
4,2006,2006 Winter Olympics,49,Skeleton,"Skeleton, Men",historico


### 3.3 Dataset: atletas_por_sexo

Análise de participação e medalhas por gênero ao longo do tempo.

In [15]:
# Histórico: inferir gênero a partir do nome do evento (Men/Women)
df_hist_sex = bronze_dataframes["olympics_historico"].copy()
df_hist_sex['year'] = df_hist_sex['edition'].str.extract(r'(\d{4})').astype(int)

def infer_gender(event_str):
    """Infere gênero a partir do nome do evento."""
    event_lower = str(event_str).lower()
    if 'women' in event_lower or 'ladies' in event_lower:
        return 'Female'
    elif 'men' in event_lower or "men's" in event_lower:
        return 'Male'
    else:
        return 'Mixed/Unknown'

df_hist_sex['gender'] = df_hist_sex['event'].apply(infer_gender)
df_hist_sex['has_medal'] = df_hist_sex['medal'].notna().astype(int)

# Agregar por ano e gênero
hist_gender = df_hist_sex.groupby(['year', 'edition', 'gender']).agg(
    total_participations=('athlete', 'count'),
    unique_athletes=('athlete_id', 'nunique'),
    total_medals=('has_medal', 'sum'),
).reset_index()
hist_gender['source'] = 'historico'

print(f"Participação por gênero (histórico): {len(hist_gender)} registros")

# Paris 2024
df_2024_sex = bronze_dataframes["olympics_paris2024"].copy()
paris_gender = df_2024_sex.groupby(['gender']).agg(
    total_participations=('name', 'count'),
    unique_athletes=('code_athlete', 'nunique'),
    total_medals=('is_medallist', 'sum'),
).reset_index()
paris_gender['year'] = 2024
paris_gender['edition'] = '2024 Summer Olympics'
paris_gender['source'] = 'paris2024'

print(f"Participação por gênero (Paris 2024): {len(paris_gender)} registros")

# Unificar
common = ['year', 'edition', 'gender', 'total_participations', 'unique_athletes', 'total_medals', 'source']
df_genero = pd.concat([hist_gender[common], paris_gender[common]], ignore_index=True)
df_genero = df_genero.drop_duplicates().reset_index(drop=True)

print(f"\natletas_por_sexo: {len(df_genero)} registros")

# Salvar
prata_path = PRATA_DIR / "atletas_por_sexo.parquet"
df_genero.to_parquet(prata_path, index=False, engine='pyarrow')
print(f"✓ Salvo: {prata_path.relative_to(DATALAKE_DIR)}")

df_genero.head(10)

Participação por gênero (histórico): 160 registros
Participação por gênero (Paris 2024): 2 registros

atletas_por_sexo: 162 registros
✓ Salvo: prata\atletas_por_sexo.parquet


,year,edition,gender,total_participations,unique_athletes,total_medals,source
0,1896,1896 Summer Olympics,Male,619,243,151,historico
1,1900,1900 Summer Olympics,Female,28,22,7,historico
2,1900,1900 Summer Olympics,Male,3460,1823,485,historico
3,1900,1900 Summer Olympics,Mixed/Unknown,832,356,142,historico
4,1904,1904 Summer Olympics,Female,17,6,10,historico
5,1904,1904 Summer Olympics,Male,2744,1221,490,historico
6,1904,1904 Summer Olympics,Mixed/Unknown,326,182,0,historico
7,1906,1906 Intercalated Games,Male,23,8,0,historico
8,1908,1908 Summer Olympics,Female,52,49,12,historico
9,1908,1908 Summer Olympics,Male,3876,2288,849,historico


In [16]:
# Metadados da camada Prata
prata_metadata = {
    "medalhas_1986_2024": {
        "dataset_name": "medalhas_1986_2024",
        "layer": "prata",
        "source_datasets": ["bronze/medal_tally_historico", "bronze/medals_paris2024"],
        "join_logic": "UNION dos quadros de medalhas histórico e Paris 2024 com colunas comuns (year, edition, country_noc, gold, silver, bronze, total). Paris 2024 foi agregado por country_code.",
        "derived_fields": ["Nenhum campo derivado adicional"],
        "data_quality_notes": "Deduplicação aplicada. Dados de 2024 agregados a partir de medalhas individuais.",
        "schema": get_schema_info(df_medalhas),
        "row_count": len(df_medalhas),
        "processing_date": PROCESSING_DATE,
    },
    "modalidades_1986_2024": {
        "dataset_name": "modalidades_1986_2024",
        "layer": "prata",
        "source_datasets": ["bronze/olympics_historico", "bronze/olympics_paris2024"],
        "join_logic": "UNION de sport/event únicos do histórico (athlete_event_result) com discipline/event de Paris 2024 (medallists). Renomeado 'discipline' para 'sport' na unificação.",
        "derived_fields": ["year (extraído da edição)", "source (historico/paris2024)"],
        "data_quality_notes": "Deduplicação aplicada. Nomes de esportes podem diferir entre fontes.",
        "schema": get_schema_info(df_modalidades),
        "row_count": len(df_modalidades),
        "processing_date": PROCESSING_DATE,
    },
    "atletas_por_sexo": {
        "dataset_name": "atletas_por_sexo",
        "layer": "prata",
        "source_datasets": ["bronze/olympics_historico", "bronze/olympics_paris2024"],
        "join_logic": "UNION de agregações por gênero/ano. Gênero inferido de nomes de eventos (Men/Women/Ladies) no histórico e campo 'gender' em 2024.",
        "derived_fields": ["gender (inferido do evento no histórico)", "has_medal", "unique_athletes", "total_participations", "total_medals"],
        "data_quality_notes": "Gênero do histórico é inferido e pode conter 'Mixed/Unknown' para eventos mistos. Deduplicação aplicada.",
        "schema": get_schema_info(df_genero),
        "row_count": len(df_genero),
        "processing_date": PROCESSING_DATE,
    },
}

for name, meta in prata_metadata.items():
    save_metadata(meta, PRATA_DIR / f"{name}_metadata.json")

print("\n✓ Camada PRATA concluída.")

  ✓ Metadata salvo: prata\medalhas_1986_2024_metadata.json
  ✓ Metadata salvo: prata\modalidades_1986_2024_metadata.json
  ✓ Metadata salvo: prata\atletas_por_sexo_metadata.json

✓ Camada PRATA concluída.


---
## 4. Camada GOLD — Analytics

Datasets analíticos prontos para consumo:
1. **Análise de Medalhas**: ranking global com visualização
2. **Análise de Modalidades**: evolução dos esportes olímpicos
3. **Análise de Gênero**: evolução da participação por gênero

### 4.1 Análise de Medalhas — medalhas_summary

In [17]:
# Agregar medalhas por país (total histórico + 2024)
medalhas_summary = df_medalhas.groupby('country_noc').agg(
    gold=('gold', 'sum'),
    silver=('silver', 'sum'),
    bronze=('bronze', 'sum'),
    total=('total', 'sum'),
    editions_participated=('edition', 'nunique'),
).reset_index()

# Adicionar nome do país (do histórico)
country_map = df_tally_hist.drop_duplicates('country_noc')[['country_noc', 'country']].dropna()
medalhas_summary = medalhas_summary.merge(country_map, on='country_noc', how='left')

# Ordenar e rankear
medalhas_summary = medalhas_summary.sort_values(
    ['gold', 'silver', 'bronze', 'total'], ascending=False
).reset_index(drop=True)
medalhas_summary['ranking'] = medalhas_summary.index + 1

# Reordenar colunas
medalhas_summary = medalhas_summary[[
    'ranking', 'country_noc', 'country', 'gold', 'silver', 'bronze', 'total', 'editions_participated'
]]

print(f"medalhas_summary: {len(medalhas_summary)} países")

# Salvar CSV
gold_medalhas_dir = GOLD_DIR / "analise_medalhas"
medalhas_summary.to_csv(gold_medalhas_dir / "medalhas_summary.csv", index=False)
print(f"✓ Salvo: gold/analise_medalhas/medalhas_summary.csv")

medalhas_summary.head(15)

medalhas_summary: 160 países
✓ Salvo: gold/analise_medalhas/medalhas_summary.csv


,ranking,country_noc,country,gold,silver,bronze,total,editions_participated
0,1,USA,United States,1235,1013,887,3135,54
1,2,URS,Soviet Union,473,376,355,1204,18
2,3,GER,Germany,367,390,374,1131,39
3,4,GBR,Great Britain,326,361,366,1053,49
4,5,CHN,People's Republic of China,325,258,221,804,20
5,6,FRA,France,303,334,378,1015,53
6,7,ITA,Italy,283,257,289,829,50
7,8,SWE,Sweden,220,237,251,708,54
8,9,NOR,Norway,212,189,176,577,50
9,10,JPN,Japan,206,190,224,620,37


In [18]:
# Visualização: Top 15 países por total de medalhas
top15 = medalhas_summary.head(15).copy()

fig, ax = plt.subplots(figsize=(14, 8))

x = range(len(top15))
width = 0.25

bars_gold = ax.bar([i - width for i in x], top15['gold'], width, label='Ouro', color='#FFD700')
bars_silver = ax.bar(x, top15['silver'], width, label='Prata', color='#C0C0C0')
bars_bronze = ax.bar([i + width for i in x], top15['bronze'], width, label='Bronze', color='#CD7F32')

ax.set_xlabel('País', fontsize=12)
ax.set_ylabel('Número de Medalhas', fontsize=12)
ax.set_title('Top 15 Países — Medalhas Olímpicas (1896–2024)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(top15['country_noc'], rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plot_path = gold_medalhas_dir / "medalhas_plot.png"
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Gráfico salvo: gold/analise_medalhas/medalhas_plot.png")
plt.show()

✓ Gráfico salvo: gold/analise_medalhas/medalhas_plot.png


C:\Users\Berg\AppData\Local\Temp\ipykernel_14964\672144021.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Metadados da análise de medalhas
medalhas_gold_metadata = {
    "dataset_name": "medalhas_summary",
    "layer": "gold",
    "source_datasets": ["prata/medalhas_1986_2024"],
    "kpi_definitions": {
        "ranking": "Posição do país no ranking global, ordenado por ouro > prata > bronze > total.",
        "gold": "Total acumulado de medalhas de ouro (1896-2024).",
        "silver": "Total acumulado de medalhas de prata (1896-2024).",
        "bronze": "Total acumulado de medalhas de bronze (1896-2024).",
        "total": "Soma de todas as medalhas.",
        "editions_participated": "Número de edições olímpicas em que o país ganhou pelo menos 1 medalha.",
    },
    "aggregation_logic": "SUM de gold/silver/bronze/total agrupados por country_noc. COUNT DISTINCT de edition para editions_participated. Ranking por gold DESC, silver DESC, bronze DESC, total DESC.",
    "assumptions": [
        "Dados de 2024 são baseados em medalhas individuais agregadas (pode haver duplicatas por eventos em equipe contados individualmente).",
        "Códigos NOC podem mudar ao longo do tempo (ex: USSR→RUS→ROC→AIN).",
        "Ranking segue convenção olímpica: prioridade para ouro, depois prata, então bronze.",
    ],
    "outputs": ["medalhas_summary.csv", "medalhas_plot.png"],
    "schema": get_schema_info(medalhas_summary),
    "row_count": len(medalhas_summary),
    "processing_date": PROCESSING_DATE,
}
save_metadata(medalhas_gold_metadata, gold_medalhas_dir / "metadata.json")
print("✓ Metadata da análise de medalhas salvo.")

### 4.2 Análise de Modalidades — Evolução dos Esportes Olímpicos

In [19]:
# Número de esportes e eventos por edição
gold_mod_dir = GOLD_DIR / "analise_modalidades"
ensure_dir(gold_mod_dir)

modalidades_summary = df_modalidades.groupby(['year', 'edition']).agg(
    total_sports=('sport', 'nunique'),
    total_events=('event', 'nunique'),
).reset_index().sort_values('year')

modalidades_summary.to_csv(gold_mod_dir / "modalidades_summary.csv", index=False)
print(f"✓ Salvo: gold/analise_modalidades/modalidades_summary.csv")

# Visualização
fig, ax1 = plt.subplots(figsize=(14, 6))

# Separar Summer e Winter
summer = modalidades_summary[modalidades_summary['edition'].str.contains('Summer')]
winter = modalidades_summary[modalidades_summary['edition'].str.contains('Winter')]

ax1.plot(summer['year'], summer['total_sports'], 'o-', color='#E63946', label='Esportes (Verão)', linewidth=2)
ax1.plot(winter['year'], winter['total_sports'], 's--', color='#457B9D', label='Esportes (Inverno)', linewidth=2)

ax1.set_xlabel('Ano', fontsize=12)
ax1.set_ylabel('Número de Esportes', fontsize=12)
ax1.set_title('Evolução do Número de Esportes Olímpicos (1896–2024)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')

plt.tight_layout()
fig.savefig(gold_mod_dir / "modalidades_plot.png", dpi=150, bbox_inches='tight')
print(f"✓ Gráfico salvo: gold/analise_modalidades/modalidades_plot.png")
plt.show()

modalidades_summary.tail(10)

✓ Salvo: gold/analise_modalidades/modalidades_summary.csv
✓ Gráfico salvo: gold/analise_modalidades/modalidades_plot.png


C:\Users\Berg\AppData\Local\Temp\ipykernel_14964\1028255062.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,year,edition,total_sports,total_events
46,2006,2006 Winter Olympics,15,75
47,2008,2008 Summer Olympics,42,279
48,2010,2010 Winter Olympics,15,78
49,2012,2012 Summer Olympics,39,261
50,2014,2014 Winter Olympics,15,86
51,2016,2016 Summer Olympics,41,263
52,2018,2018 Winter Olympics,15,89
53,2020,2020 Summer Olympics,49,290
54,2022,2022 Winter Olympics,15,94
55,2024,2024 Summer Olympics,45,288


In [20]:
# Metadados da análise de modalidades
modalidades_gold_metadata = {
    "dataset_name": "modalidades_summary",
    "layer": "gold",
    "source_datasets": ["prata/modalidades_1986_2024"],
    "kpi_definitions": {
        "total_sports": "Número de esportes distintos por edição olímpica.",
        "total_events": "Número de eventos distintos por edição olímpica.",
    },
    "aggregation_logic": "COUNT DISTINCT de sport e event agrupados por year/edition.",
    "assumptions": [
        "Nomes de esportes e eventos podem variar entre fontes.",
        "Dados de 2024 refletem apenas eventos com medalhistas.",
    ],
    "outputs": ["modalidades_summary.csv", "modalidades_plot.png"],
    "schema": get_schema_info(modalidades_summary),
    "row_count": len(modalidades_summary),
    "processing_date": PROCESSING_DATE,
}
save_metadata(modalidades_gold_metadata, gold_mod_dir / "metadata.json")
print("✓ Metadata da análise de modalidades salvo.")

  ✓ Metadata salvo: gold\analise_modalidades\metadata.json
✓ Metadata da análise de modalidades salvo.


### 4.3 Análise de Gênero — Evolução da Participação por Sexo

In [21]:
gold_gen_dir = GOLD_DIR / "analise_genero"
ensure_dir(gold_gen_dir)

# Filtrar apenas Male e Female para visualização
df_gen_viz = df_genero[df_genero['gender'].isin(['Male', 'Female'])].copy()

# Separar por tipo de Olimpíada
df_gen_viz_summer = df_gen_viz[df_gen_viz['edition'].str.contains('Summer', na=False)]

# Agregar por ano e gênero (para Summer)
genero_summary = df_gen_viz_summer.groupby(['year', 'gender']).agg(
    total_participations=('total_participations', 'sum'),
    unique_athletes=('unique_athletes', 'sum'),
    total_medals=('total_medals', 'sum'),
).reset_index()

genero_summary.to_csv(gold_gen_dir / "genero_summary.csv", index=False)
print(f"✓ Salvo: gold/analise_genero/genero_summary.csv")

# Visualização: participação por gênero ao longo do tempo (Summer)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for gender, color in [('Male', '#1D3557'), ('Female', '#E63946')]:
    mask = genero_summary['gender'] == gender
    data = genero_summary[mask].sort_values('year')
    ax1.plot(data['year'], data['unique_athletes'], 'o-', color=color, label=gender, linewidth=2, markersize=3)

ax1.set_xlabel('Ano', fontsize=11)
ax1.set_ylabel('Atletas Únicos', fontsize=11)
ax1.set_title('Atletas por Gênero — Jogos de Verão', fontsize=13, fontweight='bold')
ax1.legend()

# Proporção feminina
pivot = genero_summary.pivot_table(index='year', columns='gender', values='unique_athletes', aggfunc='sum').fillna(0)
if 'Female' in pivot.columns and 'Male' in pivot.columns:
    pivot['pct_female'] = pivot['Female'] / (pivot['Female'] + pivot['Male']) * 100
    ax2.fill_between(pivot.index, pivot['pct_female'], alpha=0.3, color='#E63946')
    ax2.plot(pivot.index, pivot['pct_female'], 'o-', color='#E63946', linewidth=2, markersize=3)
    ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('% Feminino', fontsize=11)

ax2.set_xlabel('Ano', fontsize=11)
ax2.set_title('Percentual Feminino — Jogos de Verão', fontsize=13, fontweight='bold')

plt.tight_layout()
fig.savefig(gold_gen_dir / "genero_plot.png", dpi=150, bbox_inches='tight')
print(f"✓ Gráfico salvo: gold/analise_genero/genero_plot.png")
plt.show()

genero_summary.tail(10)

✓ Salvo: gold/analise_genero/genero_summary.csv
✓ Gráfico salvo: gold/analise_genero/genero_plot.png


C:\Users\Berg\AppData\Local\Temp\ipykernel_14964\3740721286.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,year,gender,total_participations,unique_athletes,total_medals
49,2008,Female,5837,4680,920
50,2008,Male,7676,6249,1093
51,2012,Female,5785,4690,916
52,2012,Male,6996,5855,1010
53,2016,Female,6158,5028,962
54,2016,Male,7325,6111,1038
55,2020,Female,6652,5398,1113
56,2020,Male,7092,5816,1132
57,2024,Female,1162,1004,1142
58,2024,Male,1153,1050,1126


In [23]:
# Metadados da análise de gênero
genero_gold_metadata = {
    "dataset_name": "genero_summary",
    "layer": "gold",
    "source_datasets": ["prata/atletas_por_sexo"],
    "kpi_definitions": {
        "total_participations": "Total de participações em eventos (um atleta em múltiplos eventos = múltiplas participações).",
        "unique_athletes": "Número de atletas distintos.",
        "total_medals": "Total de medalhas conquistadas.",
    },
    "aggregation_logic": "SUM de participações, atletas e medalhas por year/gender. Filtrado apenas para Jogos de Verão.",
    "assumptions": [
        "Gênero do histórico é inferido a partir dos nomes de eventos (Men/Women/Ladies).",
        "Eventos mistos são excluídos da visualização.",
        "Dados de 2024 usam campo 'gender' diretamente.",
    ],
    "outputs": ["genero_summary.csv", "genero_plot.png"],
    "schema": get_schema_info(genero_summary),
    "row_count": len(genero_summary),
    "processing_date": PROCESSING_DATE,
}
save_metadata(genero_gold_metadata, gold_gen_dir / "metadata.json")
print("✓ Metadata da análise de gênero salvo.")

  ✓ Metadata salvo: gold\analise_genero\metadata.json
✓ Metadata da análise de gênero salvo.


---
## 5. metadata_schema.json — Schema Padrão de Metadados

In [24]:
metadata_schema = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Olympic Data Lake Metadata Schema",
    "description": "Schema padrão para todos os arquivos de metadados do Data Lake Olímpico.",
    "type": "object",
    "required": ["dataset_name", "layer", "processing_date"],
    "properties": {
        "dataset_name": {
            "type": "string",
            "description": "Nome identificador do dataset."
        },
        "layer": {
            "type": "string",
            "enum": ["raw", "bronze", "prata", "gold"],
            "description": "Camada do Data Lake onde o dataset reside."
        },
        "schema": {
            "type": "array",
            "description": "Lista de colunas com tipos e contagem de não-nulos.",
            "items": {
                "type": "object",
                "properties": {
                    "column": {"type": "string"},
                    "dtype": {"type": "string"},
                    "non_null_count": {"type": "integer"}
                }
            }
        },
        "source": {
            "type": "string",
            "description": "Fonte original dos dados."
        },
        "source_datasets": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Datasets de origem (para camadas derivadas)."
        },
        "description": {
            "type": "string",
            "description": "Descrição do dataset."
        },
        "main_fields": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Campos principais do dataset."
        },
        "transformations_applied": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Lista de transformações aplicadas."
        },
        "join_logic": {
            "type": "string",
            "description": "Lógica de JOIN utilizada (camadas prata/gold)."
        },
        "derived_fields": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Campos derivados/calculados."
        },
        "data_quality_notes": {
            "type": "string",
            "description": "Notas sobre qualidade dos dados."
        },
        "quality_checks": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Validações de qualidade aplicadas."
        },
        "lineage": {
            "type": "object",
            "description": "Informações de linhagem do dataset.",
            "properties": {
                "upstream": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "downstream": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            }
        },
        "kpi_definitions": {
            "type": "object",
            "description": "Definições de KPIs (camada gold)."
        },
        "aggregation_logic": {
            "type": "string",
            "description": "Lógica de agregação utilizada."
        },
        "assumptions": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Premissas e considerações."
        },
        "row_count": {
            "type": "integer",
            "description": "Número de linhas no dataset."
        },
        "column_count": {
            "type": "integer",
            "description": "Número de colunas no dataset."
        },
        "collection_date": {
            "type": "string",
            "description": "Data de coleta dos dados originais."
        },
        "processing_date": {
            "type": "string",
            "description": "Data/hora de processamento."
        },
        "observations": {
            "type": "string",
            "description": "Observações adicionais."
        },
        "outputs": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Arquivos de saída produzidos."
        }
    }
}

schema_path = DATALAKE_DIR / "metadata_schema.json"
save_metadata(metadata_schema, schema_path)
print(f"✓ metadata_schema.json salvo em: {schema_path.relative_to(DATALAKE_DIR)}")

  ✓ Metadata salvo: metadata_schema.json
✓ metadata_schema.json salvo em: metadata_schema.json


---
## 6. Copiar Notebook para Camada Gold

In [ ]:
# Copiar este notebook para a pasta gold/analise_medalhas (conforme requisito)
notebook_src = BASE_DIR / "datalake_pipeline.ipynb"
notebook_dst = GOLD_DIR / "analise_medalhas" / "notebook.ipynb"

if notebook_src.exists():
    shutil.copy2(notebook_src, notebook_dst)
    print(f"✓ Notebook copiado para: gold/analise_medalhas/notebook.ipynb")
else:
    print(f"⚠ Notebook fonte não encontrado em {notebook_src}. Copie manualmente após salvar.")

---
## 7. Validação Final — Verificação da Estrutura

In [28]:
# Verificar que todos os artefatos foram criados
print("="*60)
print("VALIDAÇÃO FINAL — Estrutura do Data Lake")
print("="*60)

expected_structure = {
    "README.md": DATALAKE_DIR / "README.md",
    "metadata_schema.json": DATALAKE_DIR / "metadata_schema.json",
    # RAW
    "raw/olympics_historico.csv": RAW_DIR / "olympics_historico.csv",
    "raw/olympics_paris2024.csv": RAW_DIR / "olympics_paris2024.csv",
    "raw/olympics_historico.json": RAW_DIR / "olympics_historico.json",
    "raw/olympics_paris2024.json": RAW_DIR / "olympics_paris2024.json",
    # BRONZE
    "bronze/olympics_historico.parquet": BRONZE_DIR / "olympics_historico.parquet",
    "bronze/olympics_paris2024.parquet": BRONZE_DIR / "olympics_paris2024.parquet",
    "bronze/medal_tally_historico.parquet": BRONZE_DIR / "medal_tally_historico.parquet",
    # PRATA
    "prata/medalhas_1986_2024.parquet": PRATA_DIR / "medalhas_1986_2024.parquet",
    "prata/modalidades_1986_2024.parquet": PRATA_DIR / "modalidades_1986_2024.parquet",
    "prata/atletas_por_sexo.parquet": PRATA_DIR / "atletas_por_sexo.parquet",
    # GOLD
    "gold/analise_medalhas/medalhas_summary.csv": GOLD_DIR / "analise_medalhas" / "medalhas_summary.csv",
    "gold/analise_medalhas/medalhas_plot.png": GOLD_DIR / "analise_medalhas" / "medalhas_plot.png",
    "gold/analise_medalhas/metadata.json": GOLD_DIR / "analise_medalhas" / "metadata.json",
    "gold/analise_modalidades/modalidades_summary.csv": GOLD_DIR / "analise_modalidades" / "modalidades_summary.csv",
    "gold/analise_modalidades/metadata.json": GOLD_DIR / "analise_modalidades" / "metadata.json",
    "gold/analise_genero/genero_summary.csv": GOLD_DIR / "analise_genero" / "genero_summary.csv",
    "gold/analise_genero/metadata.json": GOLD_DIR / "analise_genero" / "metadata.json",
}

all_ok = True
for label, path in expected_structure.items():
    exists = path.exists()
    status = "✓" if exists else "✗"
    if not exists:
        all_ok = False
    print(f"  {status} {label}")

print(f"\n{'='*60}")
if all_ok:
    print("✅ TODOS OS ARTEFATOS CRIADOS COM SUCESSO!")
else:
    print("⚠ ALGUNS ARTEFATOS NÃO FORAM ENCONTRADOS — VERIFIQUE ERROS ACIMA.")
print(f"{'='*60}")

VALIDAÇÃO FINAL — Estrutura do Data Lake
  ✓ README.md
  ✓ metadata_schema.json
  ✓ raw/olympics_historico.csv
  ✓ raw/olympics_paris2024.csv
  ✓ raw/olympics_historico.json
  ✓ raw/olympics_paris2024.json
  ✓ bronze/olympics_historico.parquet
  ✓ bronze/olympics_paris2024.parquet
  ✓ bronze/medal_tally_historico.parquet
  ✓ prata/medalhas_1986_2024.parquet
  ✓ prata/modalidades_1986_2024.parquet
  ✓ prata/atletas_por_sexo.parquet
  ✓ gold/analise_medalhas/medalhas_summary.csv
  ✓ gold/analise_medalhas/medalhas_plot.png
  ✓ gold/analise_medalhas/metadata.json
  ✓ gold/analise_modalidades/modalidades_summary.csv
  ✓ gold/analise_modalidades/metadata.json
  ✓ gold/analise_genero/genero_summary.csv
  ✓ gold/analise_genero/metadata.json

✅ TODOS OS ARTEFATOS CRIADOS COM SUCESSO!


In [27]:
# Resumo de tamanhos de arquivo por camada
print("\n📊 Resumo de Armazenamento por Camada:")
print("-"*45)
for layer_name, layer_dir in [("raw", RAW_DIR), ("bronze", BRONZE_DIR), ("prata", PRATA_DIR), ("gold", GOLD_DIR)]:
    total_size = sum(f.stat().st_size for f in layer_dir.rglob('*') if f.is_file())
    file_count = sum(1 for f in layer_dir.rglob('*') if f.is_file())
    print(f"  {layer_name:8s}: {file_count:3d} arquivos | {total_size / 1024 / 1024:.2f} MB")

total = sum(f.stat().st_size for f in DATALAKE_DIR.rglob('*') if f.is_file())
print(f"  {'TOTAL':8s}: {sum(1 for f in DATALAKE_DIR.rglob('*') if f.is_file()):3d} arquivos | {total / 1024 / 1024:.2f} MB")
print("\n🏁 Pipeline concluído com sucesso!")


📊 Resumo de Armazenamento por Camada:
---------------------------------------------
  raw     :  12 arquivos | 135.32 MB
  bronze  :  10 arquivos | 9.66 MB
  prata   :   6 arquivos | 0.06 MB
  gold    :  10 arquivos | 0.35 MB
  TOTAL   :  40 arquivos | 145.40 MB

🏁 Pipeline concluído com sucesso!
